# 02 — Preprocessing & train/test split

This notebook turns the raw reviews into model-ready text and fixes the evaluation protocol. It follows directly from the decisions reached in `01_eda`:

- **Two sentiment framings** derived from the star rating — binary (1-2 = negative, 4-5 = positive, 3 dropped) and ternary (3 = neutral) — so both can be reported, binary leading.
- **A chronological train/test split**, not a random one, because the review volume is concentrated in recent years and a deployed model would always score *new* reviews from patterns learned on *older* ones. The split is done within each category so both categories appear in train and test.

**Output:** a cached `data/processed/reviews_clean.parquet` consumed by notebooks 03 and 04, so the (one-off) spaCy preprocessing is not repeated downstream.

In [1]:
import sys
from pathlib import Path

# Resolve project root whether launched from the repo root or notebooks/.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'
RANDOM_STATE = 42

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
MODELS_DIR = PROJECT_ROOT / 'models'
for d in (PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

NAVY, BLUE, ORANGE = '#1E3A6E', '#5B8DBF', '#D97757'

In [2]:
from src import dataset, preprocessing

## 1. Load and label

Both categories are loaded, tagged, dated, and labelled under both framings in one call.

In [3]:
df = dataset.load_reviews(RAW_DIR)
print(f'Total reviews: {len(df):,}')
print(df['category'].value_counts(), '\n')
print('Binary  :', df['sentiment_binary'].value_counts(dropna=False).to_dict())
print('Ternary :', df['sentiment_ternary'].value_counts().to_dict())
df[['rating', 'category', 'text', 'sentiment_binary', 'sentiment_ternary']].head(3)

Total reviews: 87,713
category
Magazine_Subscriptions    71497
Subscription_Boxes        16216
Name: count, dtype: int64 

Binary  : {'positive': 63223, 'negative': 17705, None: 6785}
Ternary : {'positive': 63223, 'negative': 17705, 'neutral': 6785}


,rating,category,text,sentiment_binary,sentiment_ternary
0,1,Subscription_Boxes,Absolutely useless nonsense and a complete was...,negative,negative
1,2,Subscription_Boxes,"With a couple of the items, I wasn't quite sur...",negative,negative
2,1,Subscription_Boxes,Two SMALL stuffed animals and 2 little bags of...,negative,negative


**Reading.** The label counts reproduce the EDA: a strong positive skew, a moderate ~3.6:1 positive-to-negative ratio under the binary framing, and a small 3-star neutral class (~8%) under the ternary framing. The `None` entries in the binary column are exactly the 3-star reviews, dropped only for the binary task.

## 2. Text preprocessing

`clean_text` strips HTML/URLs and non-letters; `tokenize_and_lemmatize` lowercases, removes stopwords and lemmatises. VADER in RQ2 deliberately works on the **raw** text, so this cleaned column is for the TF-IDF classifiers in RQ1 only.

In [4]:
# Before/after on a couple of reviews
for t in df['text'].dropna().iloc[[0, 5]]:
    print('RAW  :', t[:90])
    print('CLEAN:', preprocessing.preprocess_review(t)[:90], '\n')

RAW  : Absolutely useless nonsense and a complete waste of money. Kitty didn't like any of the it
CLEAN: absolutely useless nonsense complete waste money kitty didn like item 

RAW  : My cats used to love them
CLEAN: cat love 



In [5]:
# Batched preprocessing of the whole corpus (uses spaCy nlp.pipe under the hood).
df['clean_text'] = preprocessing.preprocess_corpus(df['text'].tolist(), batch_size=2000)
n_empty = (df['clean_text'].str.len() == 0).sum()
print(f'Preprocessed {len(df):,} reviews; {n_empty} became empty after cleaning (no usable tokens).')
df[['text', 'clean_text']].head(3)

Preprocessed 87,713 reviews; 275 became empty after cleaning (no usable tokens).


,text,clean_text
0,Absolutely useless nonsense and a complete was...,absolutely useless nonsense complete waste mon...
1,"With a couple of the items, I wasn't quite sur...",couple item wasn sure intend application cheap...
2,Two SMALL stuffed animals and 2 little bags of...,small stuff animal little bag treat nope nope ...


**Reading.** A small number of reviews reduce to an empty string (they consisted only of punctuation, digits, or stopwords). They carry no bag-of-words signal and are dropped before modelling in notebook 03. Everything else retains its lemmatised content tokens.

## 3. Chronological train/test split

Oldest 80% for training, newest 20% for testing — applied within each category.

In [6]:
train_df, test_df = dataset.chronological_split(df, test_size=0.2, by_category=True)
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}\n')
for name, part in [('TRAIN', train_df), ('TEST', test_df)]:
    print(name, 'by category:', part['category'].value_counts().to_dict())
print()
print('Train dates:', train_df['date'].min().date(), '->', train_df['date'].max().date())
print('Test  dates:', test_df['date'].min().date(), '->', test_df['date'].max().date())

Train: 70,169  |  Test: 17,544

TRAIN by category: {'Magazine_Subscriptions': 57197, 'Subscription_Boxes': 12972}
TEST by category: {'Magazine_Subscriptions': 14300, 'Subscription_Boxes': 3244}

Train dates: 2001-10-26 -> 2021-09-03
Test  dates: 2019-10-05 -> 2023-08-30


**Reading.** Because the split is chronological within each category, the test set is dominated by the most recent reviews while both categories remain represented on both sides — the cross-category comparison RQ2 needs is preserved, and the evaluation mimics deployment rather than rewarding a model for memorising the time period.

## 4. Cache the processed data

In [7]:
keep = ['rating', 'title', 'text', 'clean_text', 'category', 'date', 'timestamp',
        'verified_purchase', 'helpful_vote', 'review_length',
        'sentiment_binary', 'sentiment_ternary']
df[keep].to_parquet(PROC_DIR / 'reviews_clean.parquet', index=False)
print('Saved ->', PROC_DIR / 'reviews_clean.parquet')

Saved -> /Users/ivelinayaneva/Desktop/Seminar/data/processed/reviews_clean.parquet


**Takeaways for notebooks 03 and 04.**
- One cached, lemmatised corpus (`reviews_clean.parquet`) with both label columns.
- A reproducible chronological, per-category split function (`dataset.chronological_split`).
- The empty-after-cleaning reviews are flagged and filtered at model time, not silently.